# Modelo de Predicción de Riesgo Social (SRI - Risk Level)

Este proyecto tiene como objetivo construir un modelo de Machine Learning capaz de estimar el nivel de riesgo social por país y año.

El análisis se basa en indicadores demográficos, económicos, educativos, de salud y laborales, con el fin de generar un **Índice de Riesgo Social (SRI- Risk Level )**.

El modelo permitirá:
- Identificar países o periodos con alto riesgo social
- Analizar tendencias socioeconómicas
- Apoyar la toma de decisiones basadas en datos

In [0]:
from pyspark.sql import functions as F

df = df.withColumn(
    "label",
    F.when(F.col("risk_level") == "HIGH", 1).otherwise(0)
)

In [0]:
df.groupBy("label").count().show()

In [0]:
exclude_cols = ["label", "risk_level", "country", "last_year"]

feature_cols = [
    c for c in df.columns
    if c not in exclude_cols
]

In [0]:
numeric_cols = [c[0] for c in df.dtypes if c[1] in ("int", "double", "float", "bigint")]

feature_cols = [c for c in numeric_cols if c != "label"]

In [0]:
data = df.select(*feature_cols, "label").dropna()

In [0]:
data = df.select(*feature_cols, "label").dropna()

In [0]:
train_df, test_df = data.randomSplit([0.8, 0.2], seed=42)

In [0]:
X_train = train_df.select(feature_cols)
y_train = train_df.select("label")

X_test = test_df.select(feature_cols)
y_test = test_df.select("label")

X_train_pd = X_train.toPandas()
y_train_pd = y_train.toPandas().values.ravel()

X_test_pd = X_test.toPandas()
y_test_pd = y_test.toPandas().values.ravel()

In [0]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)

In [0]:
model.fit(X_train_pd, y_train_pd)

In [0]:
y_pred = model.predict(X_test_pd)
y_proba = model.predict_proba(X_test_pd)

In [0]:
from sklearn.metrics import roc_auc_score

print("shape proba:", y_proba.shape)

if y_proba.shape[1] == 2:
    y_score = y_proba[:, 1]
else:
    y_score = y_proba[:, 0]

roc = roc_auc_score(y_test_pd, y_score)
print("ROC-AUC:", roc)

In [0]:
# =========================
# 📦 IMPORTS
# =========================
import pandas as pd
import numpy as np

# =========================
# 🔍 ASEGURAR TIPO CORRECTO
# =========================
# Si vienen como numpy array, los convertimos a pandas Series
y_train_pd = pd.Series(y_train_pd)
y_test_pd = pd.Series(y_test_pd)

# =========================
# 📊 DISTRIBUCIÓN DE CLASES
# =========================
print("📊 Distribución en TRAIN:")
print(y_train_pd.value_counts())

print("\n📊 Distribución en TEST:")
print(y_test_pd.value_counts())

# Normalizado (muy útil para ver desbalance)
print("\n📊 Proporción en TRAIN:")
print(y_train_pd.value_counts(normalize=True))

print("\n📊 Proporción en TEST:")
print(y_test_pd.value_counts(normalize=True))

# =========================
# 🔍 VALIDACIÓN DE CLASES ÚNICAS
# =========================
print("\n🔎 Clases en TRAIN:", np.unique(y_train_pd))
print("🔎 Clases en TEST:", np.unique(y_test_pd))

# Resultados finales

El modelo genera:

- Predicciones de riesgo social por país y año
- Clasificación de alto/bajo riesgo
- Tabla final almacenada en Databricks: `predicciones_riesgo_social`

Este output puede ser utilizado para dashboards o análisis posteriores.